**Agente**

El agente recibe una pregunta del alumno sobre la tabla Titanic y decide si necesita consultar datos reales en Spark antes de responder.
- Recibe una pregunta del alumno sobre la tabla Titanic.
- Analiza el texto de la pregunta para detectar qué necesita el usuario.
- Decide si debe usar una herramienta:
- > Si pregunta por columnas, variables o esquema → usa la herramienta de esquema.
- > Si pregunta por medias, tasas o estadísticas → usa la herramienta de estadísticas.
- > Si no necesita datos concretos → no usa herramienta.
- Consulta Spark si hace falta:
- > tool_titanic_schema() obtiene las columnas de la tabla.
- > tool_titanic_stats() calcula métricas básicas de Titanic.
- Construye un contexto con:
- > la pregunta original;
- > la herramienta utilizada;
- > los datos obtenidos desde Spark.
- Crea un prompt para el LLM indicando que debe responder como un agente docente.
- Llama al modelo GenAI de Databricks usando el cliente compatible con OpenAI.
- Genera una respuesta didáctica para el alumno, usando los datos reales si se han consultado.
- Registra todo el proceso en MLflow mediante trazas:
- > entrada del usuario;
- > herramienta usada;
- > salida de la herramienta;
- > llamada al modelo;
- > respuesta final.
- Permite visualizar el flujo en Databricks

**Instalación de librerias**

In [0]:
%pip install -U "mlflow[databricks]>=3.1.0" openai
dbutils.library.restartPython()

**Configuración inicial**

In [0]:
import json
import re
import mlflow

from openai import OpenAI
from pyspark.sql.functions import col, lower, trim, when

TABLA_DATOS_ ="cat.datos.titanic_raw_pipeline_2026_04_29"

**Conectar MLflow y el modelo**

In [0]:
# ============================================
# CONFIGURACIÓN DE MLFLOW
# ============================================

mlflow.set_tracking_uri("databricks")

# Este debe ser el experimento que has creado en:
# Experiments -> GenAI apps & agents
mlflow.set_experiment("/Shared/demo_agente_titanic_genai")

# Activa trazas automáticas de llamadas OpenAI
mlflow.openai.autolog()


# ============================================
# CONFIGURACIÓN DEL CLIENTE OPENAI PARA DATABRICKS
# ============================================

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
host = f"https://{workspace_url}"

token = (
    dbutils.notebook
    .entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

client = OpenAI(
    api_key=token,
    base_url=f"{host}/serving-endpoints"
)

# Modelo recomendado para una demo sencilla.
# Si no está disponible en tu workspace, cambia este nombre por otro endpoint disponible.
MODEL_NAME = "databricks-meta-llama-3-3-70b-instruct"

**Tools del Agente**

In [0]:
@mlflow.trace(name="tool_titanic_schema")
def tool_titanic_schema():
    """
    Herramienta que devuelve las columnas disponibles en la tabla Titanic.
    """
    df = spark.table(TABLA_DATOS_)

    return {
        "table": TABLA_DATOS_,
        "columns": df.columns
    }


@mlflow.trace(name="tool_titanic_stats")
def tool_titanic_stats():
    """
    Herramienta que calcula estadísticas básicas de la tabla Titanic.
    """
    df = spark.table(TABLA_DATOS_)

    stats = (
        df.selectExpr(
            "count(*) as num_filas",
            "avg(Survived) as tasa_supervivencia",
            "avg(Age) as edad_media",
            "avg(Fare) as tarifa_media"
        )
        .toPandas()
        .iloc[0]
        .to_dict()
    )

    return stats

**LLamar al LLM**

In [0]:
def llamar_llm(system_prompt, user_prompt, temperature=0.2, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    return response.choices[0].message.content

**Agente sencillo**

Este agente hace tres cosas:

- Mira la pregunta.
- Decide qué herramienta usar.
- Responde usando el resultado de esa herramienta.

Para no complicarlo demasiado, la decisión de herramienta la hacemos con reglas simples.

In [0]:
@mlflow.trace(name="agente_titanic_docente")
def agente_titanic_docente(pregunta):
    """
    Agente sencillo para explicar datos del Titanic a alumnos.
    """

    pregunta_lower = pregunta.lower()

    # ============================================
    # 1. Decidir qué herramienta usar
    # ============================================

    if any(palabra in pregunta_lower for palabra in ["columna", "columnas", "variable", "variables", "campos", "esquema"]):
        herramienta_usada = "tool_titanic_schema"
        contexto = tool_titanic_schema()

    elif any(palabra in pregunta_lower for palabra in ["media", "promedio", "tasa", "supervivencia", "filas", "estadística", "estadisticas", "fare", "age"]):
        herramienta_usada = "tool_titanic_stats"
        contexto = tool_titanic_stats()

    else:
        herramienta_usada = "ninguna"
        contexto = {
            "info": "No se ha usado ninguna herramienta. Responde de forma general y didáctica."
        }

    # ============================================
    # 2. Crear prompt para el modelo
    # ============================================

    system_prompt = """
Eres un agente docente para alumnos de Data Science.

Tu objetivo es responder de forma clara, breve y didáctica.
Si recibes datos en el contexto, úsalos.
No inventes métricas ni columnas que no aparezcan en el contexto.
Explica como si estuvieras en una clase práctica.
"""

    user_prompt = f"""
Pregunta del alumno:
{pregunta}

Herramienta usada:
{herramienta_usada}

Contexto:
{json.dumps(contexto, ensure_ascii=False, default=str)}
"""

    # ============================================
    # 3. Llamar al modelo
    # ============================================

    respuesta = llamar_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=0.2,
        max_tokens=500
    )

    return respuesta

**Probar el agente**

In [0]:
preguntas = [
    "¿Qué columnas tiene la tabla Titanic?",
    "¿Cuál es la tasa media de supervivencia?",
    "Explícame para qué sirve convertir la variable Sex en numérica.",
]

for pregunta in preguntas:
    print("=" * 80)
    print("PREGUNTA:")
    print(pregunta)
    print()
    
    respuesta = agente_titanic_docente(pregunta)
    
    print("RESPUESTA:")
    print(respuesta)
    print()

PREGUNTA:
¿Qué columnas tiene la tabla Titanic?

RESPUESTA:
La tabla Titanic tiene las siguientes columnas:

1. **PassengerId**: Identificador único del pasajero.
2. **Survived**: Indica si el pasajero sobrevivió (1) o no (0).
3. **Pclass**: Clase del pasajero (1, 2 o 3).
4. **Name**: Nombre del pasajero.
5. **Sex**: Género del pasajero (masculino o femenino).
6. **Age**: Edad del pasajero.
7. **SibSp**: Número de hermanos o cónyuges a bordo.
8. **Parch**: Número de padres o hijos a bordo.
9. **Ticket**: Número de ticket del pasajero.
10. **Fare**: Precio del ticket.
11. **Cabin**: Número de cabina del pasajero.
12. **Embarked**: Puerto de embarque (C, Q o S).
13. **_rescued_data**: Columna que parece contener datos rescatados o adicionales.

Estas son las columnas que tenemos disponibles en la tabla Titanic. ¿Quieres saber más sobre alguna de ellas o cómo podemos utilizarlas para analizar los datos?

PREGUNTA:
¿Cuál es la tasa media de supervivencia?

RESPUESTA:
La tasa media de super

[Trace(trace_id=tr-f9e2deeb0407d9269602f3ba8b43e86e), Trace(trace_id=tr-c1341e93d25d10dc3e202a73f8f09487), Trace(trace_id=tr-f948f47ff0884da31e261607fa56ebec)]